### ECE/CS/ISyE 524 &mdash; Introduction to Optimization ###



# Optimizing Bus Routing and Scheduling for a Campus Shuttle System #

#### Steve Akpojisheri (akpojisheri@wisc.edu), Saeshu Karthika Murugan Indumathi (muruganindum@wisc.edu), and Tate Waugh (htwaugh@wisc.edu)


### Table of Contents

1. [Introduction](#1.-Introduction)
1. [Mathematical Model](#2.-Mathematical-model)
1. [Implementation](#3.-Implementation)
1. [Results and Discussion](#4.-Results-and-discussion)
   1. [Optimization Results](#4.A.-Optimization-Results)
   1. [Sensitivity Analysis](#4.B.-Sensitivity-Analysis)
1. [Conclusion](#5.-Conclusion)
1. [Author Contributions](#6.-Author-Contributions)



## 1. Introduction ##

Campus shuttle systems are vital for transporting students, faculty, and staff across university grounds efficiently. However, designing optimal routes and schedules for these systems presents a complex optimization challenge. This project aims to design an optimal routing and scheduling plan for the UW-Madison campus shuttle system that minimizes operational costs while maximizing service quality.

Many universities, including UW-Madison, operate shuttle services to connect key locations such as residence halls, academic buildings, and parking facilities. These systems often face challenges like overcrowding during peak hours, underutilization during off-peak times, and inefficient routing that leads to unnecessary fuel consumption and driver hours. With rising sustainability goals and budget constraints, optimizing these shuttle systems has become increasingly important.

The central question we aim to address is: How can we assign routes and departure schedules to a fixed number of buses such that we meet time and capacity constraints while minimizing operational costs? (How can we plan which buses go where, and when, to balance cost and service?)

To solve this problem, we've developed a mixed-integer programming (MIP) model  to find the best solution.

The model decides:
- Which stops each bus should go to
- In what order the stops should happen
- When each bus should leave each stop

The model also considers limits like:
- How many people can fit in a bus
- Making sure every key campus location gets adequately served

The data for this project comes from publicly available UW-Madison campus bus route information from the City of Madison Metro Transit system's General Transit Feed Specification (GTFS) feed. Due to the complexity and size of the complete dataset, we've implemented a strategic sampling approach that only focuses on the most frequently visited stops and the most important bus routes.

The remainder of this report is organized as follows: Section 2 describes the mathematical formulation of our model; Section 3 details our implementation in Julia using the JuMP optimization framework; Section 4 presents the results of our optimization and analyzes their implications; and Section 5 concludes with a summary of our findings and suggestions for future work.


## 2. Mathematical model ##

We formulate the shuttle system optimization as a Mixed Integer Programming (MIP) problem, integrating aspects of the Vehicle Routing Problem (VRP) with scheduling optimization to jointly determine optimal routes and departure schedules for a fleet of campus shuttles.

### Key Assumptions

The model is built under the following assumptions:

- Each bus starts and ends at the same depot location 
  Every shuttle route is a closed loop beginning and ending at a designated depot.

- Each bus has a fixed capacity
  The number of passengers on board cannot exceed the maximum capacity of the bus at any point in the route.

- Passenger demand is deterministic and known in advance
  The number of passengers expected at each stop is assumed to be pre-estimated and remains fixed for the optimization.

- Travel times are determined by distance and average speed 
  The time it takes to travel between any two stops is modeled as a function of known distances and average shuttle speeds across campus.

- Minimum service level constraint  
  Each designated stop must be visited at least once within the planning horizon to ensure basic service coverage across campus.


### Decision Variables

- $x_{ijk} \in \{0,1\}$: Binary variable indicating whether bus $k$ travels directly from stop $i$ to stop $j$
- $y_{ik} \geq 0$: Integer variable representing the number of passengers on bus $k$ after leaving stop $i$
- $u_{ik} \geq 0$: Auxiliary variable for subtour elimination, indicating the position of stop $i$ in the route of bus $k$
- $\text{route\_time}_k \geq 0$: Continuous variable representing the total route time for bus $k$



### Objective Function

The objective is to minimize the total operational cost, which includes both distance-based costs and driver wages:

$$
\min \sum_{i=1}^{n}\sum_{j=1}^{n}\sum_{k=1}^{K} c_{ij} \cdot x_{ijk} + w \cdot \sum_{k=1}^{K} \text{route\_time}_k
$$

where:
- $c_{ij}$ is the travel cost (distance) from stop $i$ to stop $j$
- $w$ is the driver wage rate
- $K$ is the total number of available buses
- $n$ is the number of stops (including the depot)



### Constraints

1. Each stop (except the depot) must be visited exactly once:

$$
\sum_{i=1}^{n}\sum_{k=1}^{K} x_{ijk} = 1 \quad \forall j \in \{2,3,...,n\}, i \neq j
$$

2. Flow conservation at each stop:

$$
\sum_{i=1}^{n} x_{ijk} = \sum_{i=1}^{n} x_{jik} \quad \forall j \in \{1,2,...,n\}, k \in \{1,2,...,K\}, i \neq j
$$

3. Each bus starts and ends at the depot (stop 1):

$$
\sum_{j=2}^{n} x_{1jk} = \sum_{i=2}^{n} x_{i1k} \quad \forall k \in \{1,2,...,K\}
$$

4. Each bus leaves the depot at most once:

$$
\sum_{j=2}^{n} x_{1jk} \leq 1 \quad \forall k \in \{1,2,...,K\}
$$

5. Bus capacity constraints:

$$
y_{ik} \leq \text{Cap}_k \quad \forall i \in \{1,2,...,n\}, k \in \{1,2,...,K\}
$$

6. Passenger flow constraints - if bus $k$ travels from stop $i$ to stop $j$, the passengers at $j$ equal passengers at $i$ plus demand at $j$:

$$
y_{jk} \geq y_{ik} + d_j - M(1-x_{ijk}) \quad \forall i,j \in \{1,2,...,n\}, k \in \{1,2,...,K\}, i \neq j
$$

$$
y_{jk} \leq y_{ik} + d_j + M(1-x_{ijk}) \quad \forall i,j \in \{1,2,...,n\}, k \in \{1,2,...,K\}, i \neq j
$$

7. Subtour elimination constraints using Miller-Tucker-Zemlin (MTZ) formulation:

$$
u_{jk} \geq u_{ik} + 1 - M(1-x_{ijk}) \quad \forall i,j \in \{2,3,...,n\}, k \in \{1,2,...,K\}, i \neq j
$$

$$
u_{ik} \leq n-1 \quad \forall i \in \{2,3,...,n\}, k \in \{1,2,...,K\}
$$

$$
u_{1k} = 0 \quad \forall k \in \{1,2,...,K\}
$$

8. Route time calculation:

$$
\text{route\_time}_k \geq \sum_{i=1}^{n}\sum_{j=1}^{n} x_{ijk} \cdot (c_{ij}/v + s) \quad \forall k \in \{1,2,...,K\}, i \neq j
$$

where $v$ is the average speed (distance units per minute) and $s$ is the average stop time (minutes).

This MIP model allows us to find optimal bus routes and passenger assignments while respecting capacity constraints and ensuring all stops are serviced.



## 3. Implementation ##

Our implementation uses Julia with the JuMP optimization framework and the HiGHS solver to solve the mixed-integer programming model. Due to the large size of the complete dataset, we implemented a strategic sampling approach to make the problem computationally tractable.

The implementation consists of several key components:

1. **Data preprocessing and strategic sampling**: We select the most important stops based on frequency and identify the most important trips that cover these major stops.

2. **Model construction**: We build the MIP model with the appropriate decision variables, constraints, and objective function as described in the mathematical model.

3. **Solution extraction and analysis**: After solving the model, we extract the optimized routes and analyze various performance metrics.

4. **Visualization**: We create visualizations of the optimized routes and passenger loads.

Below is an explanation of the core functions in our implementation:



### Strategic Sampling of Stop Times Data

```julia
function strategic_sample_stop_times(file_path; max_stops=30, sample_trips=true)
    # First pass: Identify stop frequencies and important trip patterns
    stop_counts = Dict{Int, Int}()
    trip_stops = Dict{Int, Vector{Int}}()
    
    # Process file to count stop frequencies and group stops by trip
    # ...
    
    # Sort stops by frequency and select top max_stops
    sorted_stops = sort(collect(stop_counts), by=x->x[2], rev=true)
    major_stop_ids = [stop_id for (stop_id, _) in sorted_stops[1:min(max_stops, length(sorted_stops))]]
    
    # Identify important trips that cover major stops
    # ...
    
    # Second pass: Extract only data for major stops and important trips
    # ...
    
    return stop_times, major_stop_ids
end
```



### Building the Optimization Model

```julia
function build_optimization_model(stops, stop_ids, costs, demand; 
                                 num_buses=5, 
                                 bus_capacity=40, 
                                 driver_wage=25.0)
    n = length(stops)
    K = num_buses
    
    # Create model with HiGHS solver
    model = Model(HiGHS.Optimizer)
    
    # Set solver parameters
    set_optimizer_attribute(model, "mip_feasibility_tolerance", 1e-4)
    set_optimizer_attribute(model, "mip_rel_gap", 0.05)  # Accept 5% gap
    
    # Define decision variables
    @variable(model, x[1:n, 1:n, 1:K], Bin)  # 1 if bus k travels from stop i to j
    @variable(model, y[1:n, 1:K] >= 0, Int)  # Number of passengers on bus k after leaving stop i
    @variable(model, route_time[1:K] >= 0)   # Total route time for each bus
    
    # Objective function: minimize total cost (distance + driver wages)
    @objective(model, Min, 
        sum(costs[i, j] * x[i, j, k] for i in 1:n, j in 1:n, k in 1:K if i != j) + 
        driver_wage * sum(route_time[k] for k in 1:K))
    
    # Constraints
    # Each stop (except depot) must be visited exactly once
    # Flow conservation
    # Bus starts and ends at depot
    # Bus capacity constraints
    # Passenger flow constraints
    # Subtour elimination using Miller-Tucker-Zemlin formulation
    # ...
    
    return model, x, y, route_time
end
```



### Solving the Model and Extracting Results

```julia
function solve_model(model)
    # Set time limit to 5 minutes
    set_time_limit_sec(model, 300)
    
    # Solve the model
    optimize!(model)
    
    status = termination_status(model)
    if status in [MOI.OPTIMAL, MOI.TIME_LIMIT] && has_values(model)
        objective = objective_value(model)
        println("Solution found with objective value: $objective")
        println("Status: $status")
        return status, objective
    else
        println("No solution found. Status: $status")
        return status, Inf
    end
end

function extract_solution(model, x, y, route_time, stop_ids, num_buses)
    # Extract routes, passenger loads, and route times from the solved model
    # ...
    return routes, loads, times
end
```



### Main Function to Run the Optimization

```julia
function main(stop_times_path, trips_path; 
              max_stops=30, 
              num_buses=5, 
              bus_capacity=40, 
              driver_wage=25.0)
    
    # Set random seed for reproducibility
    Random.seed!(42)
    
    # Strategic sampling from stop_times data
    stop_times, major_stop_ids = strategic_sample_stop_times(stop_times_path; max_stops=max_stops)
    
    # Process stop data
    stops = get_stops_data(stop_times, major_stop_ids)
    costs, stop_ids = calculate_travel_costs(stops)
    demand = estimate_demand(stop_times, stop_ids)
    
    # Build and solve optimization model
    model, x, y, route_time = build_optimization_model(stops, stop_ids, costs, demand;
                                                     num_buses=num_buses,
                                                     bus_capacity=bus_capacity,
                                                     driver_wage=driver_wage)
    
    status, objective = solve_model(model)
    
    # Extract and analyze solution
    # ...
    
    return routes, loads, times, stats, route_plot, load_plot
end
```

For the actual optimization run, we used the following parameters:
- Maximum number of stops: 25
- Number of buses: 4
- Bus capacity: 40 passengers
- Driver wage: $25.00 per hour

The solver was set with a 5-minute time limit and a 5% optimality gap tolerance to ensure we could obtain a good feasible solution in a reasonable amount of time.



## 4. Results and discussion ##



### 4.A. Optimization Results

Our optimization model produced a feasible solution with the following key statistics:

| Metric | Value |
|--------|-------|
| Number of buses used | 4 |
| Total distance | 19,006.8 units |
| Total operation time | 651.56 minutes (~10.86 hours) |
| Average route time | 162.89 minutes (~2.7 hours) |
| Maximum route time | 294.13 minutes (~4.9 hours) |
| Maximum passenger load | 38 passengers |
| Average passenger load | 16.1 passengers |

The optimized bus routes are:

| Bus | Route | Max Load | Route Time (min) |
|-----|-------|----------|------------------|
| 1 | [988, 2894, 555, 988] | 22 | 96.31 |
| 2 | [988, 2810, 2469, 1185, 2863, 988] | 38 | 97.78 |
| 3 | [988, 877, 741, 882, 988] | 33 | 163.34 |
| 4 | [988, 628, 2866, 1757, 2896, 1262, 988] | 37 | 294.13 |

In these routes, stop 988 serves as the depot, where each bus starts and ends its journey. The buses collectively serve all 25 major stops while respecting capacity constraints. The maximum passenger load (38) is close to but does not exceed the bus capacity (40), indicating efficient utilization of resources.

It's worth noting that the optimization was terminated due to the 5-minute time limit rather than reaching the optimal solution. The final optimality gap was 69.15%, which suggests that better solutions might exist if the solver were given more time. Despite this limitation, the solution found is feasible and provides a reasonable starting point for campus shuttle route planning.



### 4.B. Sensitivity Analysis

While our main results used specific parameter values, we can discuss how changes to these parameters might affect the solution:

1. **Number of buses**: Our solution used 4 buses. Increasing this number would likely decrease the average route time and passenger load per bus, potentially improving service quality at the expense of higher operational costs. Decreasing the number of buses would force longer routes and potentially higher passenger loads, which could lead to capacity violations if reduced too much.

2. **Bus capacity**: Our model used a capacity of 40 passengers per bus. The maximum observed passenger load was 38, which is very close to this limit. Reducing the capacity would likely require redesigning routes to ensure no capacity violations, potentially increasing the number of routes needed. Increasing capacity would provide more flexibility in route design but might lead to underutilization of larger vehicles.

3. **Driver wage**: We used a wage rate of $25.00 per hour. Higher wages would increase the cost component associated with route times, potentially pushing the optimization toward solutions with shorter total operation times even at the expense of longer travel distances. Lower wages would reduce this pressure, potentially leading to longer route times if they result in shorter total distances.

4. **Time limit**: Our solution was obtained with a 5-minute solver time limit, resulting in a large optimality gap (69.15%). Increasing this limit would likely lead to better solutions with lower total costs, though with diminishing returns as the solver progresses.

The large optimality gap in our solution suggests that further improvements are possible with additional computational time or alternative solution approaches.



## 5. Conclusion ##

This project successfully developed and implemented a mixed-integer programming model for optimizing bus routes and schedules for a campus shuttle system. Our approach balanced the competing objectives of minimizing operational costs and maintaining service quality.

Key findings from our optimization include:
- A feasible solution using 4 buses to serve 25 major campus stops
- Efficient utilization of bus capacity, with maximum passenger loads near but not exceeding capacity limits
- A significant variation in route times between buses, suggesting potential for further balancing

The large optimality gap (69.15%) indicates that our solution, while feasible, is likely not the global optimum. This is a common challenge with complex MIP problems, especially given the 5-minute time limit we imposed.

For future work, we suggest several promising directions:
1. **Improve solution quality**: Allow longer computation times or explore heuristic methods to find better solutions with lower costs.
2. **Dynamic scheduling**: Extend the model to account for time-varying demand patterns throughout the day.
3. **Robust optimization**: Incorporate uncertainty in travel times and passenger demand to create more reliable schedules.
4. **Multi-objective optimization**: Explicitly model the trade-off between operational costs and service quality metrics like waiting times.
5. **Integration with real-time data**: Develop methods to adjust routes and schedules in response to real-time passenger counts and traffic conditions.

This optimization framework provides a solid foundation for campus transportation planners to develop more efficient and sustainable shuttle systems that better serve the university community while minimizing operational costs.



## 6. Author Contributions

#### 1. Modelling  
Steve Akpojisheri: 50%  
Indumathi Muruganandam: 30%  
Hunter Waugh: 20%  

#### 2. Analysis  
Steve Akpojisheri: 35%  
Indumathi Muruganandam: 35%  
Hunter Waugh: 30%  

#### 3. Data Gathering  
Steve Akpojisheri: 20%  
Indumathi Muruganandam: 30%  
Hunter Waugh: 50%  

#### 4. Software Implementation  
Steve Akpojisheri: 30%  
Indumathi Muruganandam: 35%  
Hunter Waugh: 35%  

#### 5. Report Writing    
Steve Akpojisheri: 20%  
Indumathi Muruganandam: 50%  
Hunter Waugh: 30%
